# DE2 — Assignment 2: Text — Inverted Index

 **Partie 0 : Setup**

In [1]:
# ============================================================
# PARTIE 0 : SETUP
# ============================================================

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
import time
import pathlib
import json
import os
from datetime import datetime

# Créer la session Spark
spark = SparkSession.builder \
    .appName("de2-assignment2") \
    .master("local[*]") \
    .getOrCreate()

print("=" * 60)
print("DE2 — Lab 2 Assignment : Inverted Index")
print("=" * 60)
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"Track: A (Esports)")
print("=" * 60)

# Créer les répertoires de sortie
pathlib.Path("outputs/lab2/inverted_index").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab2/inverted_index_csv").mkdir(parents=True, exist_ok=True)
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)

print("\n Répertoires créés")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/11 13:20:58 WARN Utils: Your hostname, Wandaogo, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/11 13:20:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 13:21:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


DE2 — Lab 2 Assignment : Inverted Index
Spark version: 4.0.1
Spark UI: http://10.255.255.254:4040
Track: A (Esports)

 Répertoires créés


**Partie 1 : Corpus Ingestion**

Étape 1a : Créer un Corpus de Test

D'abord, créez des données texte (corpus Esports) :

In [2]:
# ============================================================
# PARTIE 1 : CORPUS INGESTION
# ============================================================

# Créer un corpus texte d'exemple (Esports)
corpus_data = [
    ("doc_001", "The Finals is an esports tournament featuring the best teams in the world"),
    ("doc_002", "League of Legends Championship is the most watched esports event"),
    ("doc_003", "Counter-Strike Global Offensive remains a dominant esports title"),
    ("doc_004", "Dota 2 International is the premier esports championship with huge prize pools"),
    ("doc_005", "Valorant esports has grown rapidly since its launch in 2020"),
    ("doc_006", "The esports industry generates billions in revenue annually"),
    ("doc_007", "Professional esports players train like traditional athletes"),
    ("doc_008", "Esports tournaments are broadcast on major streaming platforms"),
    ("doc_009", "Team Liquid dominated esports in multiple game titles"),
    ("doc_010", "Esports betting and sponsorships drive the competitive gaming ecosystem"),
]

# Définir le schéma
schema = StructType([
    StructField("doc_id", StringType(), False),
    StructField("text", StringType(), False),
])

# Créer le DataFrame
df_corpus = spark.createDataFrame(corpus_data, schema=schema)

print("=" * 60)
print("CORPUS INGESTION")
print("=" * 60)
print(f"\n Corpus chargé")
print(f" Nombre de documents: {df_corpus.count()}")

print("\n Schéma du corpus:")
df_corpus.printSchema()

print("\n Premiers documents:")
df_corpus.show(3, truncate=False)

CORPUS INGESTION

 Corpus chargé


 Nombre de documents: 10

 Schéma du corpus:
root
 |-- doc_id: string (nullable = false)
 |-- text: string (nullable = false)


 Premiers documents:
+-------+-------------------------------------------------------------------------+
|doc_id |text                                                                     |
+-------+-------------------------------------------------------------------------+
|doc_001|The Finals is an esports tournament featuring the best teams in the world|
|doc_002|League of Legends Championship is the most watched esports event         |
|doc_003|Counter-Strike Global Offensive remains a dominant esports title         |
+-------+-------------------------------------------------------------------------+
only showing top 3 rows


Étape 1b (Alternative) : Charger depuis un Fichier

Si vous avez un fichier corpus :

In [3]:
# ============================================================
# ALTERNATIVE : CHARGER DEPUIS UN FICHIER
# ============================================================

import pathlib

# 1. Créer le répertoire data/ s'il n'existe pas
pathlib.Path("data").mkdir(parents=True, exist_ok=True)

# 2. Créer un fichier CSV de corpus
corpus_csv_data = """doc_id,text
doc_001,The Finals is an esports tournament featuring the best teams
doc_002,League of Legends Championship is the most watched esports event
doc_003,Counter-Strike Global Offensive remains a dominant esports title
doc_004,Dota 2 International is the premier esports championship
doc_005,Valorant esports has grown rapidly since its launch"""

with open("data/corpus.csv", "w") as f:
    f.write(corpus_csv_data)

print(" Fichier corpus.csv créé dans data/")

# 3. Charger le CSV
df_corpus = spark.read \
    .schema(schema) \
    .csv("data/corpus.csv", header=True)

print(f" Corpus chargé depuis CSV")
print(f" Documents: {df_corpus.count()}")

print("\n Schéma du corpus:")
df_corpus.printSchema()

print("\n Premiers documents:")
df_corpus.show(truncate=False)

 Fichier corpus.csv créé dans data/
 Corpus chargé depuis CSV
 Documents: 5

 Schéma du corpus:
root
 |-- doc_id: string (nullable = true)
 |-- text: string (nullable = true)


 Premiers documents:
+-------+----------------------------------------------------------------+
|doc_id |text                                                            |
+-------+----------------------------------------------------------------+
|doc_001|The Finals is an esports tournament featuring the best teams    |
|doc_002|League of Legends Championship is the most watched esports event|
|doc_003|Counter-Strike Global Offensive remains a dominant esports title|
|doc_004|Dota 2 International is the premier esports championship        |
|doc_005|Valorant esports has grown rapidly since its launch             |
+-------+----------------------------------------------------------------+



In [4]:
# ============================================================
# ALTERNATIVE : CHARGER DEPUIS UN FICHIER
# ============================================================

# 1. Créer un fichier CSV de corpus
corpus_csv_data = """doc_id,text
doc_001,The Finals is an esports tournament featuring the best teams
doc_002,League of Legends Championship is the most watched esports event
doc_003,Counter-Strike Global Offensive remains a dominant esports title
doc_004,Dota 2 International is the premier esports championship
doc_005,Valorant esports has grown rapidly since its launch"""

with open("data/corpus.csv", "w") as f:
    f.write(corpus_csv_data)

# 2. Charger le CSV
df_corpus = spark.read \
    .schema(schema) \
    .csv("data/corpus.csv", header=True)

print(f" Corpus chargé depuis CSV")
print(f" Documents: {df_corpus.count()}")

 Corpus chargé depuis CSV
 Documents: 5


**Partie 2 : Text Normalization**

Normaliser le texte (lowercase, tokenization, stop-words) :

In [6]:
# ============================================================
# PARTIE 2 : TEXT NORMALIZATION
# ============================================================

# Définir les stop-words communs en anglais
stop_words = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "is", "are", "am", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "should",
    "could", "may", "might", "must", "can", "by", "from", "as", "it", "that",
    "this", "these", "those", "i", "you", "he", "she", "we", "they", "what",
    "which", "who", "when", "where", "why", "how"
}

print("\n" + "=" * 60)
print("TEXT NORMALIZATION")
print("=" * 60)

print(f"\n Stop-words count: {len(stop_words)}")
print(f"   Sample: {list(stop_words)[:10]}")

# Étape 1 : Lowercase
df_lower = df_corpus.withColumn("text_lower", F.lower(F.col("text")))

# Étape 2 : Tokenize (split par espaces et ponctuation)
df_tokens = df_lower.withColumn(
    "tokens",
    F.split(
        F.regexp_replace(F.col("text_lower"), r"[^\w\s]", ""),  # Enlever ponctuation
        r"\s+"  # Split par espaces
    )
)

# Étape 3 : Explode tokens (une ligne par token)
df_exploded = df_tokens.select(
    F.col("doc_id"),
    F.explode(F.col("tokens")).alias("token")
)

# Étape 4 : Filter empty tokens and stop-words
df_filtered = df_exploded \
    .filter(F.col("token") != "") \
    .filter(~F.col("token").isin(list(stop_words)))

print("\n Statistiques de tokenization:")
print(f"   Total tokens (avant filtre): {df_exploded.count()}")
print(f"   Tokens filtrés (après stop-words): {df_filtered.count()}")

print("\n Tokens après normalisation:")
df_filtered.show(10, truncate=False)


TEXT NORMALIZATION

 Stop-words count: 56
   Sample: ['is', 'has', 'how', 'or', 'are', 'an', 'could', 'as', 'which', 'a']

 Statistiques de tokenization:
   Total tokens (avant filtre): 44
   Tokens filtrés (après stop-words): 33

 Tokens après normalisation:
+-------+------------+
|doc_id |token       |
+-------+------------+
|doc_001|finals      |
|doc_001|esports     |
|doc_001|tournament  |
|doc_001|featuring   |
|doc_001|best        |
|doc_001|teams       |
|doc_002|league      |
|doc_002|legends     |
|doc_002|championship|
|doc_002|most        |
+-------+------------+
only showing top 10 rows


 **Partie 3 : Build Inverted Index**

Construire l'index inversé avec groupBy(token).agg(collect_list(doc_id), count(*)) :

In [7]:
# ============================================================
# PARTIE 3 : BUILD INVERTED INDEX
# ============================================================

print("\n" + "=" * 60)
print("BUILD INVERTED INDEX")
print("=" * 60)

# Construire l'index inversé
inverted_index = (df_filtered
    .groupBy("token")
    .agg(
        F.collect_list("doc_id").alias("doc_ids"),  # Liste des doc_ids contenant le token
        F.count("*").alias("freq")  # Fréquence du token
    )
    .orderBy(F.desc("freq"))  # Trier par fréquence décroissante
)

print("\n Index inversé construit")
print(f"   Unique tokens: {inverted_index.count()}")

print("\n Schéma de l'index:")
inverted_index.printSchema()

print("\n Top 10 tokens par fréquence:")
inverted_index.show(10, truncate=False)

print("\n Statistiques:")
inverted_index.select("freq").describe().show()


BUILD INVERTED INDEX

 Index inversé construit
   Unique tokens: 28

 Schéma de l'index:
root
 |-- token: string (nullable = false)
 |-- doc_ids: array (nullable = false)
 |    |-- element: string (containsNull = false)
 |-- freq: long (nullable = false)


 Top 10 tokens par fréquence:
+------------+---------------------------------------------+----+
|token       |doc_ids                                      |freq|
+------------+---------------------------------------------+----+
|esports     |[doc_001, doc_002, doc_003, doc_004, doc_005]|5   |
|championship|[doc_002, doc_004]                           |2   |
|launch      |[doc_005]                                    |1   |
|finals      |[doc_001]                                    |1   |
|title       |[doc_003]                                    |1   |
|offensive   |[doc_003]                                    |1   |
|global      |[doc_003]                                    |1   |
|legends     |[doc_002]                             

** Partie 4 : Write to Parquet & CSV**

In [9]:
# ============================================================
# PARTIE 4 : WRITE TO PARQUET & CSV (VERSION AMÉLIORÉE)
# ============================================================

import os
import shutil

print("\n" + "=" * 60)
print("WRITE INVERTED INDEX")
print("=" * 60)

# Chemins de sortie
parquet_path = "outputs/lab2/inverted_index"
csv_path = "outputs/lab2/inverted_index_csv"

# 4a : Nettoyer les anciens fichiers
print("\n Nettoyage des anciens fichiers...")
for path in [parquet_path, csv_path]:
    if os.path.exists(path):
        try:
            shutil.rmtree(path)
            print(f"    Supprimé: {path}")
        except Exception as e:
            print(f"     Erreur suppression {path}: {e}")

# 4b : Créer les répertoires
print("\n Création des répertoires...")
os.makedirs(parquet_path, exist_ok=True)
os.makedirs(csv_path, exist_ok=True)
print(f"    Répertoires créés")

# 4c : Écrire en Parquet (garde le type ARRAY)
print("\n Écriture en Parquet...")
try:
    inverted_index.coalesce(1) \
        .write \
        .mode("overwrite") \
        .parquet(parquet_path)
    print(f"    Parquet écrit dans {parquet_path}/")
    
    # Vérifier
    parquet_files = [f for f in os.listdir(parquet_path) if not f.startswith("_")]
    parquet_size = sum(os.path.getsize(os.path.join(parquet_path, f)) for f in parquet_files)
    print(f"      Files: {len(parquet_files)}")
    print(f"      Size: {parquet_size / 1024:.2f} KB")
except Exception as e:
    print(f"    Erreur Parquet: {e}")
    import traceback
    traceback.print_exc()

# 4d : Écrire en CSV (convertir ARRAY en STRING)
print("\n Écriture en CSV...")
try:
    # Convertir doc_ids (ARRAY) en string pour CSV
    inverted_index_csv = inverted_index.withColumn(
        "doc_ids",
        F.concat_ws(",", F.col("doc_ids"))  # Convertir array → string avec virgules
    )
    
    inverted_index_csv.coalesce(1) \
        .write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(csv_path)
    print(f"    CSV écrit dans {csv_path}/")
    
    # Vérifier
    csv_files = [f for f in os.listdir(csv_path) if not f.startswith("_")]
    csv_size = sum(os.path.getsize(os.path.join(csv_path, f)) for f in csv_files)
    print(f"      Files: {len(csv_files)}")
    print(f"      Size: {csv_size / 1024:.2f} KB")
except Exception as e:
    print(f"    Erreur CSV: {e}")
    import traceback
    traceback.print_exc()

print("\n Partie 4 terminée")


WRITE INVERTED INDEX

 Nettoyage des anciens fichiers...
    Supprimé: outputs/lab2/inverted_index
    Supprimé: outputs/lab2/inverted_index_csv

 Création des répertoires...
    Répertoires créés

 Écriture en Parquet...
    Parquet écrit dans outputs/lab2/inverted_index/
      Files: 3
      Size: 1.51 KB

 Écriture en CSV...
    CSV écrit dans outputs/lab2/inverted_index_csv/
      Files: 3
      Size: 0.57 KB

 Partie 4 terminée


 **Partie 5 : Query Latency**

Mesurer la latence de requête pour les lookups :

In [10]:
# ============================================================
# PARTIE 5 : QUERY LATENCY
# ============================================================

print("\n" + "=" * 60)
print("QUERY LATENCY MEASUREMENT")
print("=" * 60)

# Charger l'index depuis Parquet (simuler une requête réelle)
index_parquet = spark.read.parquet("outputs/lab2/inverted_index")
index_parquet.createOrReplaceTempView("inverted_index_v")

# Termes à chercher
search_terms = ["esports", "championship", "tournament"]

latency_results = []

print("\n Searching for terms:")

for term in search_terms:
    # Mesurer la latence
    start_time = time.time()
    
    # Requête
    result = spark.sql(f"""
        SELECT token, freq, size(doc_ids) as doc_count, doc_ids
        FROM inverted_index_v
        WHERE token = '{term}'
    """)
    
    # Force l'exécution
    result.collect()
    
    latency_ms = (time.time() - start_time) * 1000
    
    # Récupérer les résultats
    row = result.first()
    
    if row:
        print(f"\n    {term}")
        print(f"      Latency: {latency_ms:.2f} ms")
        print(f"      Frequency: {row['freq']}")
        print(f"      Docs: {row['doc_count']}")
        
        latency_results.append({
            "term": term,
            "latency_ms": latency_ms,
            "frequency": row['freq'],
            "doc_count": row['doc_count'],
            "format": "parquet"
        })
    else:
        print(f"\n    {term} not found")

# Afficher le plan d'exécution
print("\n Execution Plan (Parquet Query):")
result.explain("formatted")

# Sauvegarder le plan
with open("proof/plan_query.txt", "w") as f:
    f.write("=== QUERY EXECUTION PLAN ===\n\n")
    f.write("Query: SELECT * FROM inverted_index WHERE token = 'esports'\n\n")
    f.write("Plan détaillé affiché dans la sortie console ci-dessus.\n")

print("\n Plan de requête sauvegardé dans proof/plan_query.txt")


QUERY LATENCY MEASUREMENT

 Searching for terms:

    esports
      Latency: 413.53 ms
      Frequency: 5
      Docs: 5

    championship
      Latency: 211.68 ms
      Frequency: 2
      Docs: 2

    tournament
      Latency: 103.35 ms
      Frequency: 1
      Docs: 1

 Execution Plan (Parquet Query):
== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [token#349, doc_ids#350, freq#351L]
Batched: true
Location: InMemoryFileIndex [file:/home/bibawandaogo/Data_Engineering2/Labs/Lab2/lab2_assignment/outputs/lab2/inverted_index]
PushedFilters: [IsNotNull(token), EqualTo(token,tournament)]
ReadSchema: struct<token:string,doc_ids:array<string>,freq:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [token#349, doc_ids#350, freq#351L]

(3) Filter [codegen id : 1]
Input [3]: [token#349, doc_ids#350, freq#351L]
Condition : (isnotnull(token#349) AND (token#349 = tournament))

(4) Project [codegen id : 1]
Out

**Partie 6 : Footprint Comparison**

Comparer la taille Parquet vs CSV :

In [11]:
# ============================================================
# PARTIE 6 : FOOTPRINT COMPARISON (VERSION ROBUSTE)
# ============================================================

import os

print("\n" + "=" * 60)
print("FOOTPRINT COMPARISON")
print("=" * 60)

def get_directory_size(path):
    """Calculer la taille totale d'un répertoire"""
    if not os.path.exists(path):
        print(f"     Répertoire manquant: {path}")
        return 0
    
    total_size = 0
    try:
        for dirpath, dirnames, filenames in os.walk(path):
            for filename in filenames:
                if not filename.startswith("_"):
                    filepath = os.path.join(dirpath, filename)
                    try:
                        total_size += os.path.getsize(filepath)
                    except OSError as e:
                        print(f"     Erreur lecture: {filepath} - {e}")
    except Exception as e:
        print(f"    Erreur: {e}")
    
    return total_size

# Vérifier les répertoires
parquet_path = "outputs/lab2/inverted_index"
csv_path = "outputs/lab2/inverted_index_csv"

print(f"\n Vérification des répertoires...")

if not os.path.exists(parquet_path):
    print(f"    Parquet path manquant: {parquet_path}")
else:
    print(f"    Parquet path existe: {parquet_path}")

if not os.path.exists(csv_path):
    print(f"    CSV path manquant: {csv_path}")
else:
    print(f"    CSV path existe: {csv_path}")

# Taille Parquet
parquet_size_bytes = get_directory_size(parquet_path)
parquet_size_kb = parquet_size_bytes / 1024 if parquet_size_bytes > 0 else 0
parquet_size_mb = parquet_size_bytes / (1024 ** 2) if parquet_size_bytes > 0 else 0

# Taille CSV
csv_size_bytes = get_directory_size(csv_path)
csv_size_kb = csv_size_bytes / 1024 if csv_size_bytes > 0 else 0
csv_size_mb = csv_size_bytes / (1024 ** 2) if csv_size_bytes > 0 else 0

# Affichage
print("\n Storage Footprint:")

print(f"\n   Parquet:")
if os.path.exists(parquet_path):
    parquet_files = [f for f in os.listdir(parquet_path) if not f.startswith("_")]
    print(f"      Size: {parquet_size_kb:.2f} KB ({parquet_size_mb:.4f} MB)")
    print(f"      Files: {len(parquet_files)}")
else:
    print(f"       Répertoire manquant")

print(f"\n   CSV:")
if os.path.exists(csv_path):
    csv_files = [f for f in os.listdir(csv_path) if not f.startswith("_")]
    print(f"      Size: {csv_size_kb:.2f} KB ({csv_size_mb:.4f} MB)")
    print(f"      Files: {len(csv_files)}")
else:
    print(f"       Répertoire manquant")

# Comparaison (seulement si les deux existent)
if parquet_size_bytes > 0 and csv_size_bytes > 0:
    compression_ratio = csv_size_bytes / parquet_size_bytes
    savings_bytes = csv_size_bytes - parquet_size_bytes
    savings_percent = (savings_bytes / csv_size_bytes * 100)
    
    print(f"\n Comparison:")
    print(f"   Compression Ratio (CSV/Parquet): {compression_ratio:.2f}x")
    print(f"   Parquet Savings: {-savings_bytes} bytes ({-savings_percent:.1f}%)")
    print(f"   Winner: {'Parquet ' if savings_bytes < 0 else 'CSV '}")
else:
    print(f"\n Impossible de comparer - un des formats manque")
    print(f"   Assurez-vous que la Partie 4 (Write) a été exécutée correctement")


FOOTPRINT COMPARISON

 Vérification des répertoires...
    Parquet path existe: outputs/lab2/inverted_index
    CSV path existe: outputs/lab2/inverted_index_csv

 Storage Footprint:

   Parquet:
      Size: 1.51 KB (0.0015 MB)
      Files: 3

   CSV:
      Size: 0.57 KB (0.0006 MB)
      Files: 3

 Comparison:
   Compression Ratio (CSV/Parquet): 0.38x
   Parquet Savings: 958 bytes (164.0%)
   Winner: Parquet 


**Partie 7 : Evidence & Metrics**

Sauvegarder les plans et métriques :

In [12]:
# ============================================================
# PARTIE 7 : EVIDENCE & METRICS
# ============================================================

print("\n" + "=" * 60)
print("SAVING EVIDENCE & METRICS")
print("=" * 60)

# 7a : Sauvegarder le plan de construction d'index
print("\n Saving index build plan...")
inverted_index.explain("formatted")

with open("proof/plan_index_build.txt", "w") as f:
    f.write("=== INVERTED INDEX BUILD PLAN ===\n\n")
    f.write("Query: groupBy(token).agg(collect_list(doc_ids), count(*))\n\n")
    f.write("Details:\n")
    f.write("- Input: Filtered tokens with doc_ids\n")
    f.write("- Transformation: GroupBy token, collect_list doc_ids, count frequency\n")
    f.write("- Output: token, doc_ids (array), freq (long)\n\n")
    f.write("Plan détaillé affiché dans la sortie console ci-dessus.\n")

print(" Plan sauvegardé dans proof/plan_index_build.txt")

# 7b : Créer et sauvegarder le CSV de métriques
print("\n Creating metrics log...")

metrics_data = {
    "run_id": ["baseline"] * len(latency_results),
    "term": [r["term"] for r in latency_results],
    "format": [r["format"] for r in latency_results],
    "latency_ms": [r["latency_ms"] for r in latency_results],
    "frequency": [r["frequency"] for r in latency_results],
    "doc_count": [r["doc_count"] for r in latency_results],
    "parquet_size_kb": [parquet_size_kb] * len(latency_results),
    "csv_size_kb": [csv_size_kb] * len(latency_results),
    "timestamp": [datetime.now().isoformat()] * len(latency_results),
}

import pandas as pd
metrics_df = pd.DataFrame(metrics_data)

metrics_df.to_csv("lab2_metrics_log.csv", index=False)

print(" Metrics sauvegardées dans lab2_metrics_log.csv\n")
print(metrics_df)

# 7c : Sauvegarder un résumé
print("\n Summary Report:")
with open("proof/summary.txt", "w") as f:
    f.write("=== LAB 2 INVERTED INDEX - SUMMARY ===\n\n")
    f.write(f"Corpus Documents: {df_corpus.count()}\n")
    f.write(f"Total Tokens (before filtering): {df_exploded.count()}\n")
    f.write(f"Unique Tokens (after filtering): {inverted_index.count()}\n")
    f.write(f"Parquet Size: {parquet_size_kb:.2f} KB\n")
    f.write(f"CSV Size: {csv_size_kb:.2f} KB\n")
    f.write(f"Compression Ratio: {compression_ratio:.2f}x\n")
    f.write(f"\nAverage Query Latency: {sum(r['latency_ms'] for r in latency_results) / len(latency_results):.2f} ms\n")
    f.write(f"\nTop Tokens:\n")
    for i, row in enumerate(inverted_index.limit(5).collect(), 1):
        f.write(f"  {i}. {row['token']}: freq={row['freq']}, docs={len(row['doc_ids'])}\n")

print(" Summary sauvegardé dans proof/summary.txt")


SAVING EVIDENCE & METRICS

 Saving index build plan...
== Physical Plan ==
AdaptiveSparkPlan (11)
+- Sort (10)
   +- Exchange (9)
      +- ObjectHashAggregate (8)
         +- Exchange (7)
            +- ObjectHashAggregate (6)
               +- Filter (5)
                  +- Generate (4)
                     +- Project (3)
                        +- Filter (2)
                           +- Scan csv  (1)


(1) Scan csv 
Output [2]: [doc_id#33, text#34]
Batched: false
Location: InMemoryFileIndex [file:/home/bibawandaogo/Data_Engineering2/Labs/Lab2/lab2_assignment/data/corpus.csv]
ReadSchema: struct<doc_id:string,text:string>

(2) Filter
Input [2]: [doc_id#33, text#34]
Condition : ((size(split(regexp_replace(lower(text#34), [^\w\s], , 1), \s+, -1), false) > 0) AND isnotnull(split(regexp_replace(lower(text#34), [^\w\s], , 1), \s+, -1)))

(3) Project
Output [2]: [doc_id#33, split(regexp_replace(lower(text#34), [^\w\s], , 1), \s+, -1) AS tokens#70]
Input [2]: [doc_id#33, text#34]

(4) Gene

**Partie 8 : Cleanup**

In [13]:
# ============================================================
# PARTIE 8 : CLEANUP
# ============================================================

print("\n" + "=" * 60)
print("CLEANUP & COMPLETION")
print("=" * 60)

# Afficher le résumé final
print("\n Lab 2 Assignment COMPLÈTE")
print("\nFichiers générés:")
print("   outputs/lab2/inverted_index/          → Parquet index")
print("   outputs/lab2/inverted_index_csv/      → CSV index")
print("   proof/                                → Preuves")
print("   lab2_metrics_log.csv                  → Métriques")

print("\nProchaines étapes:")
print("  1. Prendre des screenshots du Spark UI")
print("  2. Rédiger le rapport d'optimisation (1 page)")
print("  3. Créer GENAI.md (AI usage)")
print("  4. Pousser sur GitHub")
print("  5. Soumettre via Google Form")

# Arrêter Spark
spark.stop()
print("\n Spark arrêté")
print("\n Lab 2 Assignment terminé!")


CLEANUP & COMPLETION

 Lab 2 Assignment COMPLÈTE

Fichiers générés:
   outputs/lab2/inverted_index/          → Parquet index
   outputs/lab2/inverted_index_csv/      → CSV index
   proof/                                → Preuves
   lab2_metrics_log.csv                  → Métriques

Prochaines étapes:
  1. Prendre des screenshots du Spark UI
  2. Rédiger le rapport d'optimisation (1 page)
  3. Créer GENAI.md (AI usage)
  4. Pousser sur GitHub
  5. Soumettre via Google Form

 Spark arrêté

 Lab 2 Assignment terminé!
